# F3b — DrawBench: chấm PickScore và tổng hợp mọi seed

Chạy notebook này **sau F3a**. GPU nào cũng được (L4 là đủ); không cần clone repo. VQAScore chấm riêng bằng **F3c** trên A100, trước hay sau F3b đều được.

**Các chỉ số** (đã chốt trong `F3_preregistration.json`):

| Vai trò | Chỉ số | Chấm ở |
|---|---|---|
| Reward dẫn hướng (chỉ báo, không làm bằng chứng) | ImageReward | F3a (Phase 2) |
| Held-out, giống bài chính | CLIPScore, HPSv2.1 | F3a (Phase 2) |
| Held-out mới | PickScore (`yuvalkirstain/PickScore_v1`, processor `laion/CLIP-ViT-H-14-laion2B-s32B-b79K`, đúng README chính thức) | **F3b** |
| Held-out mới | VQAScore với CLIP-FlanT5-XXL (mặc định của `t2v-metrics`) | F3c (A100) |

**Notebook làm gì**
- Tự tìm mọi seed đã chạy đủ cả hai run trên Drive.
- Chấm và lưu cache theo từng run (`scores/`), nên bị ngắt thì Run all lại là chạy tiếp.
- In và lưu:
  - mean ± std qua các seed;
  - Δ ghép cặp của từng seed;
  - tỉ lệ prompt mà RS-LiDAR thắng.
- Chỉ số nào chưa chấm (VQAScore trước khi chạy F3c) thì bảng bỏ qua và in "chưa chấm", không lỗi.
- Kết quả được ghi vào `F3_metrics.md` và `F3_summary.csv` (F3c ghi đè bằng bản đầy đủ).

Đặt `INCLUDE_SMOKE = True` để chấm cả các run thử 2 prompt (`_smoke`) và kiểm tra pipeline chấm.

## 1. Kiểm Tra Phần Cứng GPU & VRAM
Đảm bảo bạn đã chọn Runtime: **Runtime -> Change runtime type -> A100 (hoặc L4) GPU** và bật **High-RAM**.

In [ ]:
import torch, sys, os
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    print(f"🎮 GPU: {gpu_name}")
    print(f"💾 VRAM: {vram_gb} GB")
    if vram_gb >= 22.0:
        print("✅ Tuyệt vời! GPU có đủ VRAM để kích hoạt Full-Batch VAE Decoding tốc độ cao.")
    else:
        print("⚠️ Chú ý: GPU dưới 22GB VRAM, code sẽ tự động dùng chunking để bảo vệ bộ nhớ.")
else:
    raise RuntimeError("❌ Không phát hiện GPU CUDA! Vui lòng vào Runtime -> Change runtime type -> Chọn GPU.")

!nvidia-smi

## 2. Gắn Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
base_drive = "/content/drive/My Drive" if os.path.exists("/content/drive/My Drive") else "/content/drive/MyDrive"
DRIVE_DIR = f"{base_drive}/RS-LiDAR"
TARGET_DIR = f"{DRIVE_DIR}/Target_samples"
EXP_DRIVE = f"{DRIVE_DIR}/extra_experiments/F3_drawbench"
SCORE_DIR = f"{EXP_DRIVE}/scores"
os.makedirs(SCORE_DIR, exist_ok=True)
assert os.path.isdir(TARGET_DIR), f"Không thấy {TARGET_DIR}"
print("✅ Drive:", EXP_DRIVE)

## 3. Tìm các seed đã chạy xong và hàm tiện ích

In [ ]:
import glob, json, re
import numpy as np, pandas as pd, torch
from PIL import Image
from tqdm.auto import tqdm

INCLUDE_SMOKE = False   # True: chấm cả run thử 2 prompt (_smoke) để kiểm tra pipeline chấm
SEEDS = [100, 50, 150]
N_PROMPTS = 200
SUFS = ["", "_smoke"] if INCLUDE_SMOKE else [""]


def run_dirs(seed, suf):
    li = glob.glob(f"{TARGET_DIR}/F3DB_LiDAR_SD15_n50_DDPM100_lmbda5000_seed{seed}{suf}")
    rs = glob.glob(f"{TARGET_DIR}/F3DB_RSLiDAR_SD15_n50_sig1.0_M4_DDPM100_lmbda5000_seed{seed}{suf}")
    return (li[0] if li else None), (rs[0] if rs else None)


def prompt_ids(d):
    return sorted(int(os.path.basename(p)) for p in glob.glob(f"{d}/[0-9][0-9][0-9][0-9][0-9]")
                  if os.path.exists(f"{p}/results.json"))


CASES = []   # (seed, suf, lidar_dir, rs_dir, ids)
for seed in SEEDS:
    for suf in SUFS:
        li, rs = run_dirs(seed, suf)
        if not (li and rs):
            continue
        ids_l, ids_r = prompt_ids(li), prompt_ids(rs)
        need = 2 if suf == "_smoke" else N_PROMPTS
        if len(ids_l) < need or len(ids_r) < need or ids_l != ids_r:
            print(f"⏭ seed {seed}{suf}: chưa đủ ({len(ids_l)}/{len(ids_r)} prompt) -> bỏ qua")
            continue
        CASES.append((seed, suf, li, rs, ids_l))
        print(f"✅ seed {seed}{suf}: {len(ids_l)} prompt")
assert CASES, "Chưa có seed nào chạy xong cả hai run (chạy F3a trước)."


def images_of(d, i):
    paths = sorted(glob.glob(f"{d}/{i:05d}/samples/*.png"))
    assert len(paths) == 4, f"{d}/{i:05d}: {len(paths)} ảnh (cần 4)"
    return paths


def prompt_of(d, i):
    return json.load(open(f"{d}/{i:05d}/results.json"))["prompt"]


def cache_path(d):
    return f"{SCORE_DIR}/{os.path.basename(os.path.normpath(d))}.json"


def load_cache(d):
    f = cache_path(d)
    return json.load(open(f)) if os.path.exists(f) else {}


def save_cache(d, c):
    json.dump(c, open(cache_path(d), "w"))


def score_all(metric, fn):
    """fn(prompt, paths) -> list of 4 floats. Cache per run and metric; resumable."""
    for seed, suf, li, rs, ids in CASES:
        for d in (li, rs):
            c = load_cache(d)
            got = c.setdefault(metric, {})
            todo = [i for i in ids if str(i) not in got]
            for n, i in enumerate(tqdm(todo, desc=f"{metric} {os.path.basename(d)[:45]}")):
                got[str(i)] = [float(x) for x in fn(prompt_of(d, i), images_of(d, i))]
                if n % 20 == 19:
                    save_cache(d, c)
            save_cache(d, c)

## 4. PickScore

In [ ]:
# PickScore: đúng đoạn mã trong README chính thức (huggingface.co/yuvalkirstain/PickScore_v1)
from transformers import AutoProcessor, AutoModel
device = "cuda"
pick_processor = AutoProcessor.from_pretrained("laion/CLIP-ViT-H-14-laion2B-s32B-b79K")
pick_model = AutoModel.from_pretrained("yuvalkirstain/PickScore_v1").eval().to(device)


@torch.no_grad()
def pickscore(prompt, paths):
    images = [Image.open(p).convert("RGB") for p in paths]
    image_inputs = pick_processor(images=images, padding=True, truncation=True, max_length=77,
                                  return_tensors="pt").to(device)
    text_inputs = pick_processor(text=prompt, padding=True, truncation=True, max_length=77,
                                 return_tensors="pt").to(device)
    image_embs = pick_model.get_image_features(**image_inputs)
    image_embs = image_embs / torch.norm(image_embs, dim=-1, keepdim=True)
    text_embs = pick_model.get_text_features(**text_inputs)
    text_embs = text_embs / torch.norm(text_embs, dim=-1, keepdim=True)
    return (pick_model.logit_scale.exp() * (text_embs @ image_embs.T)[0]).cpu().tolist()


score_all("PickScore", pickscore)
del pick_model
torch.cuda.empty_cache()
print("✅ PickScore xong")

## 5. Bảng tổng hợp (VQAScore hiện khi F3c đã chạy)

In [ ]:
# Tổng hợp: mỗi prompt lấy trung bình 4 ảnh; mean ± std qua các seed; Δ ghép cặp và tỉ lệ thắng theo seed
FROM_RESULTS = {"ImageReward": "IR", "Clip-Score": "CLIP", "HumanPreference": "HPS"}
FROM_CACHE = {"PickScore": "PickScore", "VQAScore": "VQAScore"}


def has_metric(d, ids, m):
    c = load_cache(d).get(m, {})
    return all(str(i) in c for i in ids)


# Chỉ số trong cache chỉ vào bảng khi đã chấm đủ cho MỌI run của MỌI seed
READY = {m: s for m, s in FROM_CACHE.items()
         if all(has_metric(d, ids, m) for _, _, li, rs, ids in CASES for d in (li, rs))}
MISSING = [s for m, s in FROM_CACHE.items() if m not in READY]
COLS = list(FROM_RESULTS.values()) + list(READY.values())
if MISSING:
    print("⏳ Chưa chấm đủ (bảng bỏ qua):", ", ".join(MISSING), "-> PickScore: F3b; VQAScore: F3c trên A100")


def per_prompt(d, ids):
    c = load_cache(d)
    out = {}
    for m, s in FROM_RESULTS.items():
        out[s] = np.array([json.load(open(f"{d}/{i:05d}/results.json"))[m]["mean"] for i in ids], dtype=float)
    for m, s in READY.items():
        out[s] = np.array([np.mean(c[m][str(i)]) for i in ids], dtype=float)
    return out


rows, seed_rows, PP = [], [], []
level = {"LiDAR": {s: [] for s in COLS}, "RS-LiDAR": {s: [] for s in COLS}}
for seed, suf, li, rs, ids in CASES:
    A, B = per_prompt(li, ids), per_prompt(rs, ids)
    PP.append((seed, suf, li, ids, A, B))
    r = {"seed": f"{seed}{suf}", "n_prompts": len(ids)}
    for s in COLS:
        if not suf:
            level["LiDAR"][s].append(A[s].mean())
            level["RS-LiDAR"][s].append(B[s].mean())
        r[f"LiDAR {s}"] = A[s].mean()
        r[f"RS-LiDAR {s}"] = B[s].mean()
        r[f"Δ {s}"] = (B[s] - A[s]).mean()
        r[f"win% {s}"] = 100 * ((B[s] > A[s]).mean() + 0.5 * (B[s] == A[s]).mean())
    seed_rows.append(r)

per_seed = pd.DataFrame(seed_rows)
n_seeds = len(level["LiDAR"]["IR"])
for lab in ("LiDAR", "RS-LiDAR"):
    row = {"run": lab, "n_seeds": n_seeds}
    for s in COLS:
        v = np.array(level[lab][s])
        row[s] = (f"{v.mean():.3f} ± {v.std(ddof=1):.3f}" if len(v) > 1 else f"{v.mean():.3f}") if len(v) else ""
    rows.append(row)
summary = pd.DataFrame(rows)

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 50)
print("Mean (± std qua các seed) trên 200 prompt DrawBench, 4 ảnh mỗi prompt:")
display(summary)
print("Từng seed: Δ = RS-LiDAR − LiDAR (trung bình theo prompt); win% = % prompt RS-LiDAR cao hơn (hoà tính 1/2):")
display(per_seed.round(4))

# Bảng theo 11 nhóm DrawBench: chỉ để mô tả (đã chốt trước), không dùng làm kết luận.
# Δ = trung bình theo (prompt, seed); win% = % cặp (prompt, seed) mà RS-LiDAR cao hơn (hoà tính 1/2).
import hashlib
CAT_FILE = f"{EXP_DRIVE}/drawbench.jsonl"
CAT_SHA256 = "c47f23e6bb4ae883c0527cce1fb3052fc5cd8f34df9da9a7845c33996f041886"
cat_table = None
if os.path.exists(CAT_FILE):
    assert hashlib.sha256(open(CAT_FILE, "rb").read()).hexdigest() == CAT_SHA256, "drawbench.jsonl trên Drive khác bản đã chốt"
    CAT_META = [json.loads(l) for l in open(CAT_FILE, encoding="utf-8") if l.strip()]
    use = [x for x in PP if not x[1]] or PP
    recs = []
    for seed, suf, li, ids, A, B in use:
        for i in ids[:3]:
            assert prompt_of(li, i) == CAT_META[i]["prompt"], f"prompt {i} của run khác drawbench.jsonl"
        for k, i in enumerate(ids):
            recs.append({"seed": f"{seed}{suf}", "category": CAT_META[i]["category"],
                         **{s: B[s][k] - A[s][k] for s in COLS}})
    dd = pd.DataFrame(recs)
    g = dd.groupby("category")
    cat_table = pd.DataFrame({"n_prompts": g.size() // dd["seed"].nunique()})
    for s in COLS:
        cat_table[f"Δ {s}"] = g[s].mean()
        cat_table[f"win% {s}"] = g[s].apply(lambda x: 100 * ((x > 0).mean() + 0.5 * (x == 0).mean()))
    cat_table = cat_table.sort_values("n_prompts", ascending=False).reset_index()
    print(f"Theo nhóm prompt ({dd['seed'].nunique()} seed; chỉ mô tả, không dùng làm kết luận):")
    display(cat_table.round(4))
else:
    print(f"⚠️ Không thấy {CAT_FILE} (F3a ghi file này) -> bỏ qua bảng theo nhóm.")

per_seed.to_csv(f"{EXP_DRIVE}/F3_summary.csv", index=False)
md = ("# F3: DrawBench (SD 1.5, DDPM-100, σ = 1, M = 4 cố định từ Table 1; 200 prompt, 4 ảnh/prompt)\n\n"
      + summary.to_markdown(index=False) + "\n\n## Từng seed\n\n"
      + per_seed.round(4).to_markdown(index=False)
      + ("\n\n## Theo nhóm prompt (chỉ mô tả)\n\n" + cat_table.round(4).to_markdown(index=False)
         if cat_table is not None else "")
      + "\n\nIR là reward dẫn hướng (chỉ báo). Chỉ số chính: VQAScore (CLIP-FlanT5-XXL). "
        "Chỉ số phụ: CLIP, HPS, PickScore. Quy tắc diễn đạt (chốt trước): cao hơn ở mọi seed -> 'cao hơn'; "
        "thấp hơn ở mọi seed -> 'thấp hơn'; còn lại -> 'ngang nhau'. "
        "Chốt trước trong F3_preregistration.json; báo mọi chỉ số, mọi seed.\n"
      + (f"\nChưa chấm: {', '.join(MISSING)}.\n" if MISSING else ""))
open(f"{EXP_DRIVE}/F3_metrics.md", "w", encoding="utf-8").write(md)
print(f"Đã lưu {EXP_DRIVE}/F3_metrics.md và F3_summary.csv")

## Tự ngắt máy khi chạy xong
Cell cuối tự ngắt runtime Colab sau khi Run all xong (tiết kiệm compute unit). Kết quả đã lưu trên Drive.

In [ ]:
# Tự ngắt runtime khi Run all chạy xong, để không tốn thêm compute unit.
# Kết quả đã nằm trên Drive; đẩy hết dữ liệu đang chờ ghi lên Drive trước khi ngắt.
# Muốn giữ máy để xem/chạy tiếp thì đặt AUTO_DISCONNECT = False trước khi Run all.
AUTO_DISCONNECT = True
if AUTO_DISCONNECT:
    try:
        from google.colab import drive, runtime
        drive.flush_and_unmount()
        print("🛑 Đã chạy xong và lưu lên Drive. Tự ngắt runtime Colab.")
        runtime.unassign()
    except Exception as e:
        print("⚠️ Không tự ngắt được runtime, hãy tự ngắt: Runtime → Disconnect and delete runtime.", e)